# 🧥 MeshVTON: Full 3D Virtual Try-On Training

With this notebook, you can run the entire pipeline:

1. GPU and library setup
2. Mounting data from Google Drive
3. SMPL-X body parameter extraction
4. 3D garment rendering
5. Model training

Requirements: GPU runtime (T4/A100/H100)

---
## 1️⃣ GPU Control

In [1]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU not found! Runtime → Change runtime type → Select GPU')

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


---
## 2️⃣ Clone the project

In [2]:
import os

# === CHANGE THIS: Your own GitHub repo URL ===
REPO_URL = 'https://github.com/SerhanTelatar/MeshVTON.git'
PROJECT_DIR = '/content/MeshVTON'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print('✅ Repo cloned')
else:
    !cd {PROJECT_DIR} && git pull
    print('✅ Repo updated')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/MeshVTON'...
remote: Enumerating objects: 326, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 326 (delta 156), reused 268 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (326/326), 2.55 MiB | 9.55 MiB/s, done.
Resolving deltas: 100% (156/156), done.
✅ Repo cloned
Working directory: /content/MeshVTON


---
## 3️⃣ Install the Libraries

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# Drive'dan yükle — derleme yok
from google.colab import drive
drive.mount('/content/drive')

!pip install -q fvcore iopath
!pip install /content/drive/MyDrive/wheels/pytorch3d*.whl

# Core dependencies — FIXED VERSIONS for IDM-VTON compatibility
!pip install diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 peft==0.7.1 -q
!pip install -q omegaconf opencv-python-headless pillow scipy
!pip install -q lpips einops timm

# 3D Pipeline dependencies
!pip install -q smplx trimesh pyrender

# PyTorch3D
try:
    import pytorch3d
    print(f'✅ PyTorch3D: {pytorch3d.__version__}')
except:
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"

---
## 4️⃣ Connect Google Drive & Turn on Data

In [7]:
from google.colab import drive
drive.mount('/content/drive')

# === CHANGE THIS: Your folder path in Drive ===
DRIVE_DATA = '/content/drive/MyDrive/MeshVTON'

import os
print('Drive content:')
for f in os.listdir(DRIVE_DATA):
    size = os.path.getsize(os.path.join(DRIVE_DATA, f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive content:
  MeshVTON_Train.ipynb (0.1 MB)
  test_pairs.csv (0.1 MB)
  val_pairs.csv (0.1 MB)
  train_pairs.csv (1.1 MB)
  pretrained.zip (226.8 MB)
  images.zip (1817.5 MB)
  poses.zip (169.1 MB)
  segments.zip (98.5 MB)
  densepose.zip (645.0 MB)
  agnostic.zip (429.5 MB)
  garments_3d.zip (5363.9 MB)
  checkpoints (0.0 MB)
  smplx_params.zip (9.9 MB)
  pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl (63.5 MB)


In [ ]:
import zipfile
import shutil
from pathlib import Path

PROJECT = Path('/content/MeshVTON')
DRIVE = Path(DRIVE_DATA)

def extract_zip(zip_name, target_dir):
    """Extract zip from Drive (skips if not present)."""
    zip_path = DRIVE / zip_name
    if not zip_path.exists():
        print(f'  ⚠️ {zip_name} not found, skipping')
        return
    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    print(f'  📦 Extracting {zip_name} → {target_dir}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target)
    print(f'  ✅ {zip_name} completed')

for d in ['data/raw/images', 'data/processed/poses', 'data/processed/segments',
          'data/processed/densepose', 'data/processed/agnostic',
          'data/processed/smplx_params', 'data/processed/renders_3d',
          'data/processed/normal_maps', 'data/processed/depth_maps',
          'data/garments_3d', 'checkpoints/pretrained']:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)

print('📂 Extracting zip files...')
print()

extract_zip('images.zip', PROJECT / 'data/raw/images')

extract_zip('poses.zip', PROJECT / 'data/processed')
extract_zip('segments.zip', PROJECT / 'data/processed')
extract_zip('densepose.zip', PROJECT / 'data/processed')
extract_zip('agnostic.zip', PROJECT / 'data/processed')

extract_zip('garments_3d.zip', PROJECT / 'data/garments_3d')

extract_zip('pretrained.zip', PROJECT / 'checkpoints')

# --- CACHED preprocessing outputs: extract instead of recomputing ---
# (saved to Drive by the final "Save to Drive" cell on a previous run)
extract_zip('smplx_params.zip', PROJECT / 'data/processed/smplx_params')
extract_zip('renders_3d.zip',  PROJECT / 'data/processed/renders_3d')
extract_zip('normal_maps.zip', PROJECT / 'data/processed/normal_maps')
extract_zip('depth_maps.zip',  PROJECT / 'data/processed/depth_maps')

for csv_name in ['train_pairs.csv', 'val_pairs.csv', 'test_pairs.csv']:
    s = DRIVE / csv_name
    if s.exists():
        shutil.copy2(s, PROJECT / 'data/raw' / csv_name)

print()
print('✅ All data loaded!')

In [ ]:
"""Fix data paths and verify — SINGLE CELL"""
import shutil, subprocess, os
from pathlib import Path
P = Path('/content/MeshVTON')
# ============================================================
# 1. Fix nested extraction paths
# ============================================================
nested_img = P / 'data/raw/images/images'
if nested_img.exists():
    for f in nested_img.glob('*'):
        shutil.move(str(f), str(P / 'data/raw/images' / f.name))
    shutil.rmtree(nested_img, ignore_errors=True)
    print('🔧 Person images fixed')
nested_g = P / 'data/garments_3d/garments_3d'
if nested_g.exists():
    for d in nested_g.iterdir():
        dest = P / 'data/garments_3d' / d.name
        if not dest.exists():
            shutil.move(str(d), str(dest))
    shutil.rmtree(nested_g, ignore_errors=True)
    print('🔧 3D garments fixed')
smplx_target = P / 'checkpoints/pretrained/smplx'
smplx_target.mkdir(parents=True, exist_ok=True)
result = subprocess.run(['find', str(P / 'checkpoints'), '-name', '*.npz'],
                       capture_output=True, text=True)
for line in result.stdout.strip().split('\n'):
    if line.strip():
        src = Path(line.strip())
        if src.exists() and 'NEUTRAL_2020' not in src.name:
            dest = smplx_target / 'SMPLX_NEUTRAL.npz'
            if src.resolve() != dest.resolve():
                shutil.copy2(str(src), str(dest))
                print(f'🔧 SMPL-X → {dest}')
        elif src.exists() and 'NEUTRAL_2020' in src.name:
            dest = smplx_target / 'SMPLX_NEUTRAL_2020.npz'
            if src.resolve() != dest.resolve():
                shutil.copy2(str(src), str(dest))
vposer_target = P / 'checkpoints/pretrained/vposer'
vposer_target.mkdir(parents=True, exist_ok=True)
# ============================================================
# 2. Final Verification
# ============================================================
print('\n📊 Final Data Check')
print('=' * 45)
checks = {
    'Person Images':      len(list((P/'data/raw/images').glob('*.jpg'))),
    'Poses':              len(list((P/'data/processed/poses').glob('*'))),
    'Segmentation':       len(list((P/'data/processed/segments').glob('*'))),
    'DensePose':          len(list((P/'data/processed/densepose').glob('*'))),
    'Agnostic':           len(list((P/'data/processed/agnostic').glob('*'))),
    '3D Garment (upper)': len(list((P/'data/garments_3d/upper_body').glob('*'))) if (P/'data/garments_3d/upper_body').exists() else 0,
    '3D Garment (lower)': len(list((P/'data/garments_3d/lower_body').glob('*'))) if (P/'data/garments_3d/lower_body').exists() else 0,
    '3D Garment (dress)': len(list((P/'data/garments_3d/dresses').glob('*'))) if (P/'data/garments_3d/dresses').exists() else 0,
    '3D Garment (outer)': len(list((P/'data/garments_3d/outerwear').glob('*'))) if (P/'data/garments_3d/outerwear').exists() else 0,
    'SMPL-X Model':       (P/'checkpoints/pretrained/smplx/SMPLX_NEUTRAL.npz').exists(),
    'train_pairs.csv':    (P/'data/raw/train_pairs.csv').exists(),
}
all_ok = True
for name, val in checks.items():
    ok = '✅' if val else '❌'
    if not val: all_ok = False
    print(f'  {ok} {name}: {val}')
print('=' * 45)
print('🎉 ALL READY!' if all_ok else '⚠️ Some data is missing')

---
## 5️⃣ SMPL-X Body Parameter Extraction

Her kişi görüntüsünden 3D beden parametrelerini (şekil, poz) çıkarir.

In [ ]:
import os
os.chdir('/content/MeshVTON')
# __init__.py dosyalarını oluştur (Python'un modülleri bulması için)
from pathlib import Path
for d in ['src', 'src/data', 'src/data/preprocessing', 'src/modules', 'src/models']:
    init = Path(d) / '__init__.py'
    init.parent.mkdir(parents=True, exist_ok=True)
    if not init.exists():
        init.touch()
        print(f'Created {init}')
print('✅ Done — şimdi SMPL-X hücresini tekrar çalıştır')

In [ ]:
!pip install smplx
import smplx


In [ ]:
import sys, os
os.chdir('/content/MeshVTON')
sys.path.insert(0, '/content/MeshVTON')
!git pull

from pathlib import Path
smplx_out = Path('data/processed/smplx_params')
existing = list(smplx_out.glob('*')) if smplx_out.exists() else []
if existing:
    print(f'⏩ SMPL-X params already present ({len(existing)} files) — skipping extraction')
else:
    from src.data.preprocessing.extract_smplx import extract_smplx
    extract_smplx(
        image_dir='data/raw/images',
        output_dir='data/processed/smplx_params',
        model_dir='checkpoints/pretrained',
        device='cuda',
        save_mesh=True,
        mesh_dir='data/processed/smplx_meshes',
    )

---
## 6️⃣ 3D Clothing Rendering

CLOTH3D mesh'lerini SMPL-X beden modellerine giydirip 2D'ye render eder.

In [ ]:
!pip install trimesh

In [ ]:
# Reload latest code
import importlib, sys
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]
!cd /content/MeshVTON && git pull

from pathlib import Path
renders_out = Path('data/processed/renders_3d')
existing = list(renders_out.glob('*')) if renders_out.exists() else []
if existing:
    print(f'⏩ 3D renders already present ({len(existing)} files) — skipping rendering')
else:
    from src.data.preprocessing.render_garment import render_garments
    render_garments(
        garments_dir='data/garments_3d',
        smplx_params_dir='data/processed/smplx_params',
        output_dir='data/processed/renders_3d',
        pairs_csv='data/raw/train_pairs.csv',
        normal_maps_dir='data/processed/normal_maps',
        depth_maps_dir='data/processed/depth_maps',
        resolution=512,
        device='cuda',
    )

---
## 7️⃣ Start Training 🚀

In [ ]:
!cd /content/MeshVTON && git pull
import torch
from omegaconf import OmegaConf

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Pick config by VRAM (GPU name matching is unreliable, e.g. "RTX PRO 6000")
if vram_gb >= 70:          # H100/A100 80GB, RTX PRO 6000 96GB+
    TRAIN_CONFIG = {
        'batch_size': 8, 'gradient_accumulation_steps': 2,
        'learning_rate': 2e-5, 'num_epochs': 10,
        'mixed_precision': 'bf16', 'gradient_checkpointing': False,
        'save_every': 2, 'log_every': 25,
    }
elif vram_gb >= 35:        # A100 40GB, L40, etc.
    TRAIN_CONFIG = {
        'batch_size': 4, 'gradient_accumulation_steps': 4,
        'learning_rate': 2e-5, 'num_epochs': 10,
        'mixed_precision': 'bf16', 'gradient_checkpointing': True,
        'save_every': 2, 'log_every': 25,
    }
else:                       # T4 16GB or smaller
    TRAIN_CONFIG = {
        'batch_size': 1, 'gradient_accumulation_steps': 16,
        'learning_rate': 1e-5, 'num_epochs': 5,
        'mixed_precision': 'fp16', 'gradient_checkpointing': True,
        'save_every': 1, 'log_every': 50,
    }

# >>> Actually write the values into a runtime config that training will read <<<
cfg = OmegaConf.load('configs/train.yaml')
cfg.data.batch_size                         = TRAIN_CONFIG['batch_size']
cfg.training.epochs                         = TRAIN_CONFIG['num_epochs']
cfg.training.gradient_accumulation_steps    = TRAIN_CONFIG['gradient_accumulation_steps']
cfg.training.mixed_precision                = TRAIN_CONFIG['mixed_precision']
cfg.training.gradient_checkpointing         = TRAIN_CONFIG['gradient_checkpointing']
cfg.training.optimizer.lr                   = TRAIN_CONFIG['learning_rate']
cfg.training.checkpoint.save_every_n_epochs = TRAIN_CONFIG['save_every']
cfg.training.logging.log_every_n_steps      = TRAIN_CONFIG['log_every']
OmegaConf.save(cfg, 'configs/train_active.yaml')

eff = TRAIN_CONFIG['batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps']
print(f'🖥️ GPU: {gpu_name} ({vram_gb:.0f} GB)')
print(f'⚡ Effective batch size: {eff}')
print('✅ configs/train_active.yaml written:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
!cd /content/MeshVTON && PYTHONPATH=/content/MeshVTON python scripts/train.py --config configs/train_active.yaml

---
## 8️⃣ Save checkpoints to Drive

In [ ]:
import shutil
from pathlib import Path

# DRIVE_DATA değişkenini kontrol et — notebook'undaki değerle aynı olmalı
DRIVE_DATA = '/content/drive/MyDrive/MeshVTON'  # veya Shared drives path'in

src_ckpt = Path('/content/MeshVTON/checkpoints/runs')
dst_ckpt = Path(DRIVE_DATA) / 'checkpoints'
dst_ckpt.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    for f in src_ckpt.glob('*.pt'):
        shutil.copy2(f, dst_ckpt / f.name)
        print(f'✅ {f.name} → Drive\'a kaydedildi ({f.stat().st_size / 1e9:.1f} GB)')
else:
    print("⚠️ checkpoints/runs klasörü yok — eğitim çalıştı mı?")

In [ ]:
import shutil
from pathlib import Path

# Copy checkpoints to Drive (so they aren't lost if Colab closes)
src_ckpt = Path('checkpoints/runs')
dst_ckpt = Path(DRIVE_DATA) / 'checkpoints'
dst_ckpt.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    for f in src_ckpt.glob('*.pt'):
        shutil.copy2(f, dst_ckpt / f.name)
        print(f'✅ {f.name} → Saved to Drive')

# Also save preprocessing results (to avoid re-running)
for folder in ['smplx_params', 'renders_3d', 'normal_maps', 'depth_maps']:
    src = Path(f'data/processed/{folder}')
    if src.exists() and any(src.iterdir()):
        dst = Path(DRIVE_DATA) / f'{folder}.zip'
        if not dst.exists():
            shutil.make_archive(str(dst).replace('.zip',''), 'zip', src)
            print(f'✅ {folder} → Saved to Drive')

print('\n✅ All results saved to Google Drive!')

In [ ]:
import shutil
from pathlib import Path

src_ckpt = Path('/content/MeshVTON/checkpoints/runs')
dst_ckpt = Path(DRIVE_DATA) / 'checkpoints'
dst_ckpt.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    for f in src_ckpt.glob('*.pt'):
        shutil.copy2(f, dst_ckpt / f.name)
        print(f'✅ {f.name} → Saved to Drive')
else:
    print("⚠️ checkpoints/runs klasörü yok!")

---
## 9️⃣ Quick Test (Optional)

Eğitilmiş model ile tek bir görüntüde try-on test et.

In [ ]:
ls

In [ ]:
import torch
from src.models.tryon_pipeline import TryOnPipeline
from src.inference.image_tryon import ImageTryOn
from PIL import Image
import matplotlib.pyplot as plt

# 1. Pipeline'ı checkpoint'tan yükle

config = {
    "model_channels": 320,
    "controlnet_3d_channels": 9,
}
pipeline = TryOnPipeline.from_config(config)
# ControlNet3D ağırlıklarını yükle
ckpt = torch.load(actual_path, map_location='cuda', weights_only=False)
if 'model' in ckpt:
    pipeline.controlnet_3d.load_state_dict(ckpt['model'], strict=False)
elif 'controlnet_3d' in ckpt:
    pipeline.controlnet_3d.load_state_dict(ckpt['controlnet_3d'])
else:
    pipeline.load_state_dict(ckpt, strict=False)
print("✅ ControlNet3D yüklendi")
print("✅ Model yüklendi")

# 2. ImageTryOn'ı oluştur
tryon = ImageTryOn(
    pipeline=pipeline,
    config={
        "device": "cuda",
        "image": {"resolution": 512},
        "sampling": {
            "num_inference_steps": 50,
            "guidance_scale": 7.5,
            "seed": 42,
        },
    }
)

# 3. Test
person_img = 'data/raw/images/00001_00.jpg'
garment_img = 'data/raw/images/00001_00.jpg'

result = tryon.run(person_img, garment_img)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(Image.open(person_img))
plt.title('Person')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(result)
plt.title('Result')
plt.axis('off')
plt.tight_layout()
plt.savefig('test_result.png', dpi=150)
plt.show()

In [ ]:
import os

paths_to_check = [
    '/content/MeshVTON/checkpoints/runs/final.pt',
    '/content/drive/MyDrive/MeshVTON/checkpoints/final.pt',
    '/content/drive/MyDrive/MeshVTON/checkpoints/runs/final.pt'
]

print("🔍 Model dosyası aranıyor...")
found = False
for path in paths_to_check:
    if os.path.exists(path):
        print(f"✅ Dosya bulundu: {path}")
        found = True
        # Değişkeni güncelle ki bir sonraki hücrede kullanabilelim
        MODEL_PATH = path
    else:
        print(f"❌ Yok: {path}")

if not found:
    print("\n⚠️ Dosya hiçbir yerde bulunamadı! Lütfen eğitim hücresinin (8. adım) başarıyla tamamlandığından emin ol.")

In [ ]:
import torch
from src.models.tryon_pipeline import TryOnPipeline
from src.inference.image_tryon import ImageTryOn
from PIL import Image
import matplotlib.pyplot as plt
import os

# 0. Çalışma dizinini düzelt
PROJECT_DIR = '/content/MeshVTON'
os.chdir(PROJECT_DIR)
print(f"📍 Çalışma dizini: {os.getcwd()}")

# Yukarıdaki hücrede bulunan yolu kullanıyoruz, yoksa varsayılan yerel yolu dene
actual_path = globals().get('MODEL_PATH', '/content/drive/MyDrive/MeshVTON/checkpoints/runs/final.pt')

if not os.path.exists(actual_path):
    print(f"❌ HATA: {actual_path} bulunamadı!")
else:
    # 1. Pipeline'ı yükle
    config = {
        "person_in_channels": 9,
        "model_channels": 320,
        "out_channels": 4,
        "context_dim": 768,
        "controlnet_channels": 9,
    }
    pipeline = TryOnPipeline.from_config(config)

    ckpt = torch.load(actual_path, map_location='cuda', weights_only=False)
    pipeline.load_state_dict(ckpt['model'])
    print(f"✅ Model yüklendi: {actual_path}")

    # 2. ImageTryOn oluştur
    tryon = ImageTryOn(
        pipeline=pipeline,
        config={
            "device": "cuda",
            "image": {"resolution": 512},
            "sampling": {
                "num_inference_steps": 50,
                "guidance_scale": 7.5,
                "seed": 42,
            },
        }
    )

    # 3. Test (Göreceli yollar artık çalışacaktır)
    person_img = '/content/00001_00.jpg'
    if os.path.exists(person_img):
        print(f"🎨 Try-on başlatılıyor: {person_img}")
        result = tryon.run(person_img, person_img)
        plt.figure(figsize=(8, 8))
        plt.imshow(result)
        plt.axis('off')
        plt.title("DiffFit-3D Inference Result")
        plt.show()
    else:
        print(f"❌ Test görüntüsü hala bulunamadı: {os.path.abspath(person_img)}")
        print("Klasör içeriği:", os.listdir('data/raw/images')[:5] if os.path.exists('data/raw/images') else "Klasör yok!")


